# Data Quality
This notebook contains the code for data quality checks, including missing values, duplicates, types, inconsistent labels and EDA visualizations.

### Imports

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from src.data import load_raw, quality_report, label_issues, rows_to_drop, multiclass_label, MODES

## General quality check

In [ ]:
df = load_raw()
df.info()
quality_report(df).to_frame("count")

**Findings**
- 10000 rows, 14 columns, correct types.
- No missing values, as expected, no duplicate rows, no duplicate UDIs.
- `Type` always matches the first letter of `Product ID`. There are no negative values, and process temperature is never below air temperature.
> No imputation or cleaning of the features is needed.

## Label consistency

In [ ]:
issues = label_issues(df)
display(issues["reason"].value_counts())
issues

**Findings**

`Machine failure` should be 1 if and only if at least one mode is active. That rule is broken in two ways:
- **9 rows `failure_no_mode`:** a failure with no mode. The failure label is valid, but there is no mode to assign.
- **18 rows `mode_no_failure`:** RNF active but `Machine failure = 0`. These are the only true inconsistencies.

Separately, **24 rows `multiple_modes`** have more than one mode at once. They are not errors, because the modes are independent and can happen together.

## Multiclass rows and final classes

In [ ]:
multi_kept = df.drop(rows_to_drop(df, "multi"))
print(f"Multiclass: dropped {len(df) - len(multi_kept)} rows, kept {len(multi_kept)}")
multiclass_label(multi_kept).value_counts()

**Decisions**
- **Binary:** keep all 10000 rows and use `Machine failure` as given. The 18 `mode_no_failure` rows stay as no failure.
- **Multiclass (No failure, TWF, HDF, PWF, OSF):** drop 27 rows with no learnable label, the 9 `failure_no_mode` and the 18 RNF only. RNF is excluded because it is random.
- **Multi-mode rows:** instead of having both modes, the rarest one is the one given (TWF < OSF < PWF < HDF). This keeps 24 real failures and keeps these hard cases in the test set. The cost is that predicting the other true mode counts as an error, we will check this in the error analysis.

**Final classes**: 9643 No failure, 106 HDF, 95 OSF, 83 PWF, 46 TWF.